# 11 MQA_and_GQA

## Problem

MHA、MQA、GQA 分别在做什么折中？为什么 GQA 常常是工业界很现实的选择？

## Dependencies

需要理解 multi-head attention 和 KV cache 的成本来源。


## Module_Goals

- 区分 MHA、MQA、GQA 的 Q/K/V 组织方式
- 理解共享 K/V 为什么能减小 cache 开销
- 理解 GQA 为什么是表达能力和效率之间的折中
- 会写最小 MHA、MQA、GQA 形状对比代码

## Scope_And_Approach

这个 notebook 会先用直觉和小例子说明问题，再给出最小实现。重点是把模块为什么存在、输入输出是什么、关键步骤如何衔接说明清楚。


## 工程直觉

MHA 的问题不是它不好，而是它在推理时的 K/V 开销会随着 head 数增加而明显变重。

三种结构可以这样记：

- **MHA**：每个 query head 都有自己独立的 K/V head。
- **MQA**：所有 query head 共享同一组 K/V。
- **GQA**：若干个 query head 共享一组 K/V，也就是“分组共享”。

所以它们的关键不是 Q，而是 **K/V 到底共享到什么程度**。共享越多，cache 越省；但共享过度，也可能损失表达灵活性。


In [ ]:
import numpy as np

hidden_dim = 16
num_query_heads = 8
head_dim = hidden_dim // (hidden_dim // 2)  # 这里只是过渡写法，后面直接指定
head_dim = 4
seq_len = 32


def kv_cache_elements(seq_len, kv_heads, head_dim):
    return seq_len * kv_heads * head_dim * 2  # 乘 2 是因为 K 和 V 都要存

mha_kv_heads = 8
mqa_kv_heads = 1
gqa_kv_heads = 2

print('MHA cache elements =', kv_cache_elements(seq_len, mha_kv_heads, head_dim))
print('MQA cache elements =', kv_cache_elements(seq_len, mqa_kv_heads, head_dim))
print('GQA cache elements =', kv_cache_elements(seq_len, gqa_kv_heads, head_dim))


In [ ]:
# 最小张量组织
batch = 1
seq_len = 5
hidden = np.random.randn(batch, seq_len, hidden_dim)

# MHA: q_heads = 8, kv_heads = 8
Q_mha = np.random.randn(batch, seq_len, 8, head_dim)
K_mha = np.random.randn(batch, seq_len, 8, head_dim)
V_mha = np.random.randn(batch, seq_len, 8, head_dim)

# MQA: q_heads = 8, kv_heads = 1
Q_mqa = np.random.randn(batch, seq_len, 8, head_dim)
K_mqa = np.random.randn(batch, seq_len, 1, head_dim)
V_mqa = np.random.randn(batch, seq_len, 1, head_dim)

# GQA: q_heads = 8, kv_heads = 2, 每 4 个 query head 共享一组 KV
Q_gqa = np.random.randn(batch, seq_len, 8, head_dim)
K_gqa = np.random.randn(batch, seq_len, 2, head_dim)
V_gqa = np.random.randn(batch, seq_len, 2, head_dim)

print('MHA shapes:', Q_mha.shape, K_mha.shape, V_mha.shape)
print('MQA shapes:', Q_mqa.shape, K_mqa.shape, V_mqa.shape)
print('GQA shapes:', Q_gqa.shape, K_gqa.shape, V_gqa.shape)

# 把 GQA 的 kv head 映射到 query head group
query_to_group = {q_head: q_head // 4 for q_head in range(8)}
print('GQA query_to_group =', query_to_group)


## Trade_Offs_And_Risk_Points

- 把 MQA 理解成“少几个头”而不是“query head 仍然很多，但 K/V 共享”。
- 只记住 MHA/MQA/GQA 的定义，而说不出它们为什么会影响 KV cache 成本。
- 以为 GQA 只是理论花活。事实上它在很多实际系统里非常常见，因为折中合理。

## Checkpoints

- 假设 `num_query_heads=16`，自己设计一个 `kv_heads=4` 的 GQA 分组方案。
- 思考为什么 MQA cache 最省，但不一定总能保留最佳效果。
- 估算当上下文长度翻倍时，三种结构的 cache 增长会怎样变化。

## Summary

理解了 MHA、MQA、GQA，你就已经抓住了现代注意力工程优化的一条主线：不是所有头都必须为每个位置保存完整独立的 K/V。

## Next_Module

下一节进入 MLA。它会在“减少 cache”的方向上再往前走一步，把压缩思路引入注意力状态表示。
